# Restricted specimen registration for repeated PT reconstruction

A canonical PT is observed through unordered cuts of many realizations. This experiment tests a restricted specimen intercept around a source-only scFates backbone. Unknown nephron membership is integrated out; no complete nephron trajectory is reconstructed.

In [ ]:
from pathlib import Path
import hashlib, io, json, sys
import numpy as np
import pandas as pd
from scipy import sparse
from IPython.display import display
REPO=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pseudospace').is_dir())
sys.path.insert(0,str(REPO))
from pseudospace.repeated_inputs import load_repeated_mouse_inputs
from pseudospace.count_representation import fit_count_representation,transform_count_representation
from pseudospace.repeated_structure import fit_atlas
from pseudospace.trajectory_benchmark import fit_scfates_axis,fit_dpt_axis,project_neighbor_axis
from pseudospace.canonical_mean_atlas import fit_mean_atlas,project_mean_atlas
from pseudospace.specimen_registration import fit_specimen_offset
from pseudospace.within_segment_validation import fit_segment_covariate_oracle,predict_segment_covariate_oracle
from pseudospace.stage_cache import cached_payload
from importlib.metadata import version
import argparse,os
parser=argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root',default=os.environ.get('PSEUDOSPACE_DATA_ROOT',str(REPO/'data')))
parser.add_argument('--results-root',default=os.environ.get('PSEUDOSPACE_RESULTS_ROOT',str(REPO/'results')))
args,_=parser.parse_known_args()
DATA_ROOT,RESULTS_ROOT=Path(args.data_root).expanduser(),Path(args.results_root).expanduser()
OUT=RESULTS_ROOT/'repeated_structure_specimen_registration'
OUT.mkdir(parents=True,exist_ok=True)
REFERENCE_OUT=RESULTS_ROOT/'healthy_reconstruction_trajectory_benchmark'
NOTEBOOK_LOGIC_VERSION='24.specimen_registration.1'
SEED,N_COMPONENTS,N_FOLDS=15,15,3
CONTROLS=('Ctrl1A2','Ctrl1A4')
GRID=np.linspace(0,1,101)
SCFATES_PARAMS=dict(nodes=30,seed=SEED)
COUNT_ATLAS_PARAMS=dict(grid=GRID,bandwidth=.08,theta=100.,min_effective=12.)
MARKERS=set('Slc5a2 Slc5a12 Gatm Slc22a6 Slc13a3 Slc1a1 Cyp2e1 Slc7a13 Slc22a7 Slc5a1 Cyp7b1 Slc7a12 Lrp2 Cubn Slc34a1 Slc6a18 Acsm3 Agt'.casefold().split())
VERSIONS={name:version(name) for name in ['scanpy','scFates','anndata','scikit-learn']}
CODE_FILES=[REPO/'pseudospace'/f for f in ['repeated_inputs.py','count_representation.py','trajectory_benchmark.py','repeated_structure.py','canonical_mean_atlas.py','specimen_registration.py','within_segment_validation.py','stage_cache.py']]
CODE_DIGEST=hashlib.sha256(b''.join(p.read_bytes() for p in CODE_FILES)).hexdigest()
BASELINE_SCORE_PATH=REFERENCE_OUT/'scores.csv'
if not BASELINE_SCORE_PATH.is_file(): raise FileNotFoundError(f'Required notebook-21 baseline is missing: {BASELINE_SCORE_PATH}')
BASELINE_SCORE_DIGEST=hashlib.sha256(BASELINE_SCORE_PATH.read_bytes()).hexdigest()
CACHE_PARAMS=dict(version=NOTEBOOK_LOGIC_VERSION,controls=CONTROLS,seed=SEED,
    components=N_COMPONENTS,folds=N_FOLDS,markers=sorted(MARKERS),grid=GRID.tolist(),
    scfates=SCFATES_PARAMS,mean_atlas=dict(bandwidth=.08,min_effective=12.),
    count_atlas={**COUNT_ATLAS_PARAMS,'grid':GRID.tolist()},
    count_representation_theta=100.,projection_neighbors=15,gene_spline_ridge=.001,
    quarter_read_fraction=.25,quarter_read_seed_rule='seed+fold',
    baseline_score_digest=BASELINE_SCORE_DIGEST,versions=VERSIONS)
print('Installed benchmark versions:',VERSIONS)


OFFSET_PARAMS=dict(variance_fraction=.99,penalty=4.,iterations=5)
CACHE_PARAMS.pop('count_atlas')
CACHE_PARAMS.update(offset=OFFSET_PARAMS,calibration_hash='sha256(structure_id) first8 %2 ==0',evaluation_hash='complement',calibration_regimes=['full','S3_only'],oracle_degree=2,oracle_ridge=.01,dpt_neighbors=30)
reference_scores=pd.read_csv(BASELINE_SCORE_PATH)
reference_positions=pd.read_csv(REFERENCE_OUT/'positions.csv')
reference_positions=reference_positions[reference_positions.method.isin(['scFates_neighbor','DPT_neighbor'])]
PANEL_PATH=RESULTS_ROOT/'within_segment_repeated_structure_validation'/'selected_genes.csv'
frozen_panel=pd.read_csv(PANEL_PATH,usecols=['source_specimen','test_specimen','fold','segment','gene','training_cv_gain'])
if frozen_panel.duplicated(['source_specimen','fold','segment','gene']).any(): raise ValueError('Duplicate panel genes')


## Actual healthy inputs

Same validated structure filtering and fixed gene folds as notebooks 21–23. Markers and response-fold genes are excluded individually, but all-measured-gene exposure introduces aggregate compositional dependence. Depth is unused.

In [ ]:
healthy=load_repeated_mouse_inputs(DATA_ROOT,samples=CONTROLS)
obs=healthy.obs.copy()
ids=obs.structure_id.to_numpy(str)
specimens=obs['sample'].astype(str).to_numpy()
labels=obs.segment_class.astype(str).str.replace('PT-','',regex=False).to_numpy()
if set(np.unique(specimens))!=set(CONTROLS): raise ValueError('Expected exactly the two healthy control specimens')
if not np.isin(labels,['S1','S2','S3']).all(): raise ValueError('Resolve uncertain PT labels upstream')
anatomy=np.array([{'S1':0,'S2':1,'S3':2}[x] for x in labels],dtype=int)
genes=healthy.var_names.to_numpy(str)
X=healthy.layers['lognorm']; X=X.toarray().astype(np.float64,copy=False) if sparse.issparse(X) else np.asarray(X,dtype=np.float64)
raw_counts=healthy.layers['counts']
raw_total=np.asarray(raw_counts.sum(axis=1)).ravel()
if not np.isfinite(X).all() or not np.isfinite(raw_total).all() or np.any(raw_total<=0): raise ValueError('Invalid expression or all-gene exposure')
if 'n_spots' not in obs: raise ValueError('Required n_spots metadata is missing')
n_spots=pd.to_numeric(obs.n_spots,errors='coerce').to_numpy(float)
if not np.isfinite(n_spots).all() or np.any(n_spots<0): raise ValueError('Invalid n_spots metadata')
marker_mask=np.array([g.casefold() in MARKERS for g in genes])
gene_fold=np.array([int(hashlib.sha256(g.casefold().encode()).hexdigest()[:8],16)%N_FOLDS for g in genes])
manifest=dict(healthy.uns['repeated_mouse_input_manifest'])
manifest.update(notebook_logic_version=NOTEBOOK_LOGIC_VERSION,config=CACHE_PARAMS,code_digest=CODE_DIGEST,markers_excluded=sorted(MARKERS),dpt_used_for_fitting=False,depth_used=False)
(OUT/'input_manifest.json').write_text(json.dumps(manifest,indent=2))
display(pd.crosstab(specimens,labels))
print('Structures:',len(ids),'genes:',len(genes),'outer split: whole specimen')

## Preregistered calibration protocol

Hash half of the query specimen calibrates the offset; the disjoint half is evaluated. The S3-only regime uses annotation to construct a partial calibration population, then hides it from the estimator. The retained source-curve subspace captures 99% variation. Five updates and penalty 4 are fixed. Full-rank curves permit no restricted offset. This gives calibration data to the new arms, unlike zero-calibration DPT/scFates references; it is not the same information allowance. Frozen earlier gene programs retain development/selection bias.

In [ ]:
def dense_panel(matrix,rows,columns):
    value=matrix[rows][:,columns]
    return value.toarray() if sparse.issparse(value) else np.asarray(value)

def fit_gene_spline(response,z,sample):
    return fit_atlas(response,z,sample,grid=GRID,ridge=.001)

def normalized_variance(response,sample):
    mean=np.mean([response[sample==s].mean(axis=0) for s in np.unique(sample)],axis=0)
    return np.maximum(np.mean([((response[sample==s]-mean)**2).mean(axis=0) for s in np.unique(sample)],axis=0),1e-3)

def primary_rows(response,predictions,variance,query_anatomy,meta):
    out=[]
    for method,pred in predictions.items():
        scaled=(response-pred)**2/variance
        for seg,name in [(None,'all'),(0,'S1'),(1,'S2'),(2,'S3')]:
            use=np.ones(len(response),bool) if seg is None else query_anatomy==seg
            if use.any(): out.append(dict(**meta,method=method,segment=name,n=int(use.sum()),normalized_mse=float(scaled[use].mean())))
    return out


In [ ]:
def run_registration():
    scores,selected_scores,thin_scores,stability,positions,offset_rows,diagnostics,failures=[],[],[],[],[],[],[],[]
    from scipy.stats import spearmanr
    for source in CONTROLS:
        target=next(s for s in CONTROLS if s!=source)
        tr,te=specimens==source,specimens==target
        query_ids=ids[te]
        calibration=np.array([int(hashlib.sha256(x.encode()).hexdigest()[:8],16)%2==0 for x in query_ids])
        evaluation=~calibration
        if not calibration.any() or not evaluation.any() or np.any(calibration&evaluation): raise ValueError('Invalid disjoint split')
        for fold in range(N_FOLDS):
            key=dict(source_specimen=source,test_specimen=target,fold=fold)
            try:
                mg=np.flatnonzero(~marker_mask&(gene_fold!=fold));eg=np.flatnonzero(~marker_mask&(gene_fold==fold))
                C=dense_panel(raw_counts,tr,mg);Q=dense_panel(raw_counts,te,mg)
                Y,transform=fit_count_representation(C,raw_total[tr],specimens[tr],N_COMPONENTS,theta=100.,seed=SEED)
                query_Y=transform_count_representation(Q,raw_total[te],transform)
                sc_z,sc_meta=fit_scfates_axis(Y,anatomy[tr],**SCFATES_PARAMS)
                dpt_z,dpt_meta=fit_dpt_axis(Y,anatomy[tr],n_neighbors=30,seed=SEED)
                reference=fit_mean_atlas(Y,sc_z,grid=GRID,bandwidth=.08,min_effective=12)
                train_response=X[tr][:,eg];query_response=X[te][:,eg]
                variance=normalized_variance(train_response,specimens[tr])
                decoder=fit_gene_spline(train_response,sc_z,specimens[tr])
                dpt_decoder=fit_gene_spline(train_response,dpt_z,specimens[tr])
                full_z={'scFates_neighbor':project_neighbor_axis(Y,sc_z,query_Y,k=15),'DPT_neighbor':project_neighbor_axis(Y,dpt_z,query_Y,k=15)}
                full_predictions={'scFates_neighbor':decoder.predict(full_z['scFates_neighbor']),'DPT_neighbor':dpt_decoder.predict(full_z['DPT_neighbor'])}
                reproduced=pd.DataFrame(primary_rows(query_response,full_predictions,variance,anatomy[te],key))
                prior=reference_scores.query('source_specimen==@source and fold==@fold and method in ["scFates_neighbor","DPT_neighbor"]')
                joined=reproduced.merge(prior,on=['method','segment'],suffixes=('_new','_old'),validate='one_to_one')
                if len(joined)!=8 or not np.allclose(joined.normalized_mse_new,joined.normalized_mse_old,rtol=1e-8,atol=1e-10): raise ValueError('Matched DPT/scFates reference scores did not reproduce')
                for method,z in full_z.items():
                    saved=reference_positions.query('source_specimen==@source and fold==@fold and method==@method').set_index('structure_id')
                    if saved.index.duplicated().any() or set(saved.index)!=set(query_ids): raise ValueError('Reference IDs mismatch')
                    if not np.allclose(z,saved.loc[query_ids,'z'],rtol=1e-8,atol=1e-10): raise ValueError('Reference coordinates did not reproduce')

                train_cov=np.column_stack([np.log(raw_total[tr]),np.log1p(n_spots[tr]),np.log1p(np.asarray((raw_counts[tr][:,mg]>0).sum(axis=1)).ravel())])
                query_cov=np.column_stack([np.log(raw_total[te]),np.log1p(n_spots[te]),np.log1p(np.asarray((raw_counts[te][:,mg]>0).sum(axis=1)).ravel())])
                oracle=fit_segment_covariate_oracle(train_response,train_cov,anatomy[tr],specimens[tr],ridge=.01,degree=2)
                oracle_prediction=predict_segment_covariate_oracle(oracle,query_cov[evaluation],anatomy[te][evaluation])
                full=sparse.csr_matrix(raw_counts[te]);rng=np.random.default_rng(SEED+fold)
                thinned=sparse.csr_matrix((rng.binomial(full.data.astype(np.int64),.25),full.indices.copy(),full.indptr.copy()),shape=full.shape)
                library=np.asarray(thinned.sum(axis=1)).ravel().astype(float)
                thin_Y=transform_count_representation(dense_panel(thinned,np.ones(len(library),bool),mg),library,transform)[evaluation]
                eval_Y=query_Y[evaluation];eval_a=anatomy[te][evaluation];eval_ids=query_ids[evaluation]
                eval_response=query_response[evaluation]
                gene_to_j={g:j for j,g in enumerate(genes[eg])}
                panel=frozen_panel.query('source_specimen==@source and test_specimen==@target and fold==@fold')
                for regime in ['full','S3_only']:
                    # Labels construct the deliberately incomplete calibration population;
                    # neither offset estimator receives them.
                    use_calibration=calibration if regime=='full' else calibration&(anatomy[te]==2)
                    if use_calibration.sum()<12: raise ValueError('Insufficient calibration support')
                    offsets={'scFates_neighbor':np.zeros(Y.shape[1])}
                    for method,restricted in [('restricted_offset',True),('unrestricted_offset',False)]:
                        offset,meta=fit_specimen_offset(reference,query_Y[use_calibration],restricted=restricted,**OFFSET_PARAMS)
                        if restricted:
                            _,_,basis=np.linalg.svd(reference['mean']-reference['mean'].mean(axis=0),full_matrices=False)
                            if np.max(np.abs(basis[:meta['rank']]@offset),initial=0)>1e-8:raise ValueError('Offset violates source path restriction')
                            if meta['rank']==Y.shape[1] and np.any(offset!=0):raise ValueError('Full-rank restriction must be exact zero')
                        offsets[method]=offset
                        offset_rows.append(dict(**key,regime=regime,method=method,offset=json.dumps(offset.tolist()),metadata=json.dumps(meta),rank=meta['rank'],offset_norm=float(np.linalg.norm(offset)),calibration_n=int(use_calibration.sum())))
                    coords={method:project_neighbor_axis(Y,sc_z,eval_Y-offset,k=15) for method,offset in offsets.items()}
                    coords['DPT_neighbor']=full_z['DPT_neighbor'][evaluation]
                    thin_coords={method:project_neighbor_axis(Y,sc_z,thin_Y-offset,k=15) for method,offset in offsets.items()}
                    thin_coords['DPT_neighbor']=project_neighbor_axis(Y,dpt_z,thin_Y,k=15)
                    predictions={method:(dpt_decoder if method=='DPT_neighbor' else decoder).predict(z) for method,z in coords.items()}
                    predictions['quadratic_segment_coverage_oracle']=oracle_prediction
                    case={**key,'regime':regime}
                    scores.extend(primary_rows(eval_response,predictions,variance,eval_a,case))
                    thin_predictions={method:(dpt_decoder if method=='DPT_neighbor' else decoder).predict(z) for method,z in thin_coords.items()}
                    thin_scores.extend(primary_rows(eval_response,thin_predictions,variance,eval_a,case))
                    for seg,segname in enumerate(['S1','S2','S3']):
                        selected=panel[panel.segment==segname]
                        if selected.empty or not set(selected.gene).issubset(gene_to_j): raise ValueError('Invalid frozen panel '+segname)
                        jj=np.array([gene_to_j[g] for g in selected.gene],int)
                        scale=np.maximum(np.var(train_response[anatomy[tr]==seg],axis=0),1e-3)[jj]
                        use=eval_a==seg
                        if not use.any(): raise ValueError('Missing evaluation segment')
                        for method,pred in predictions.items(): selected_scores.append(dict(**case,method=method,segment=segname+'_selected',n=int(use.sum()),selected_gene_count=len(jj),normalized_mse=float(np.mean((eval_response[use][:,jj]-pred[use][:,jj])**2/scale))))
                    for method,z in coords.items():
                        movement=np.abs(z-thin_coords[method]);change=np.abs(z-coords['scFates_neighbor'])
                        for seg,name in [(None,'all'),(0,'S1'),(1,'S2'),(2,'S3')]:
                            use=np.ones(len(z),bool) if seg is None else eval_a==seg
                            stability.append(dict(**case,method=method,segment=name,n=int(use.sum()),quarter_read_mae=float(movement[use].mean()),fraction_read_moved_over_0_1=float(np.mean(movement[use]>.1)),coordinate_change_from_identity=float(change[use].mean()),rank_vs_identity=float(spearmanr(z[use],coords['scFates_neighbor'][use]).statistic)))
                        for identity,value,thin_value,seg in zip(eval_ids,z,thin_coords[method],eval_a):positions.append(dict(**case,method=method,structure_id=identity,z=float(value),thinned_z=float(thin_value),segment=['S1','S2','S3'][int(seg)]))
                diagnostics.append(dict(**key,status='complete',calibration_n=int(calibration.sum()),evaluation_n=int(evaluation.sum()),scfates_metadata=json.dumps(sc_meta),dpt_metadata=json.dumps(dpt_meta)))
                print('Registration transfer:',source,'→',target,'fold',fold,flush=True)
            except Exception as exc:
                failures.append(dict(**key,error=str(exc)));diagnostics.append(dict(**key,status='failed',calibration_n=int(calibration.sum()),evaluation_n=int(evaluation.sum()),scfates_metadata='',dpt_metadata=''))
    frames=dict(scores=scores,selected_scores=selected_scores,thin_scores=thin_scores,stability=stability,positions=positions,offsets=offset_rows,diagnostics=diagnostics,failures=failures)
    return {name:pd.DataFrame(rows).to_json(orient='table') for name,rows in frames.items()}


In [ ]:
CACHE_INPUTS=dict(X=X,counts=raw_counts,library=raw_total,ids=ids,specimens=specimens,labels=labels,n_spots=n_spots,gene_fold=gene_fold,markers=marker_mask,frozen_panel=frozen_panel,reference_scores=reference_scores,reference_positions=reference_positions)
payload=cached_payload('repeated_structure_specimen_registration',run_registration,root=OUT/'stage_cache',params=CACHE_PARAMS,inputs=CACHE_INPUTS,code=CODE_DIGEST)
outputs={name:pd.read_json(io.StringIO(str(value)),orient='table') for name,value in payload.items()}
for name,frame in outputs.items():frame.to_csv(OUT/(name+'.csv'),index=False)
if len(outputs['diagnostics'])!=6 or len(outputs['failures']) or len(outputs['offsets'])!=24:raise RuntimeError('Incomplete registration experiment; inspect failures.csv')
# Enforce the unchanged evaluation population across methods and calibration regimes.
populations=outputs['positions'].groupby(['source_specimen','fold','regime','method']).structure_id.apply(lambda x:tuple(sorted(x)))
for _,group in populations.groupby(level=['source_specimen','fold']):
    if len(group)!=8 or any(x!=group.iloc[0] for x in group):raise ValueError('Evaluation IDs changed between calibration regimes')
if not np.isfinite(outputs['positions'].z).all() or not outputs['positions'].z.between(0,1).all():raise ValueError('Invalid evaluation coordinates')
display(outputs['diagnostics']);display(outputs['offsets'][['source_specimen','fold','regime','method','rank','offset_norm','calibration_n']])
for name in ['scores','selected_scores']:
    frame=outputs[name];keys=['source_specimen','fold','regime','segment']
    baseline=frame.query('method=="scFates_neighbor"')[keys+['normalized_mse']].rename(columns={'normalized_mse':'identity_mse'})
    oracle=frame.query('method=="quadratic_segment_coverage_oracle"')[keys+['normalized_mse']].rename(columns={'normalized_mse':'oracle_mse'})
    comparison=frame.merge(baseline,on=keys,validate='many_to_one').merge(oracle,on=keys,validate='many_to_one')
    comparison['gain_vs_identity_percent']=100*(1-comparison.normalized_mse/comparison.identity_mse)
    comparison['gain_vs_oracle_percent']=100*(1-comparison.normalized_mse/comparison.oracle_mse)
    comparison.to_csv(OUT/(name+'_comparison.csv'),index=False)
    display(comparison.groupby(['source_specimen','regime','segment','method'])[['normalized_mse','gain_vs_identity_percent','gain_vs_oracle_percent']].mean())


## Prediction and coverage robustness

A geometric restriction is not proof of physical or injury direction. No arm or segment-specific parameter is selected using query errors. Healthy results do not enable AKI or human interpretation.

In [ ]:
import matplotlib.pyplot as plt
comparison=pd.read_csv(OUT/'selected_scores_comparison.csv')
fig,axes=plt.subplots(2,2,figsize=(11,7))
for column,source in enumerate(CONTROLS):
    for row,regime in enumerate(['full','S3_only']):
        ax=axes[row,column]
        for method,offset,color in [('restricted_offset',-.08,'C2'),('unrestricted_offset',.08,'C3')]:
            for i,segment in enumerate(['S1_selected','S2_selected','S3_selected']):
                values=comparison.query('source_specimen==@source and regime==@regime and method==@method and segment==@segment').gain_vs_identity_percent.to_numpy()
                ax.scatter(i+offset+np.linspace(-.03,.03,len(values)),values,color=color,label=method if i==0 else None)
                ax.plot([i+offset-.07,i+offset+.07],[values.mean()]*2,color=color)
        ax.axhline(0,color='gray',linewidth=.7);ax.set(xticks=[0,1,2],xticklabels=['S1','S2','S3'],title=source+' / '+regime+' calibration',ylabel='Program error reduction vs identity (%)')
axes[0,0].legend(frameon=False,fontsize=9)
fig.suptitle('Frozen offset; disjoint evaluation half; dots are gene folds')
fig.tight_layout();fig.savefig(OUT/'registration_program_prediction.png',dpi=160);plt.show()
